# **Stage_07_07 -  Modelo TCN (many-to-one)**

**Introducción - Temporal Convolutional Networks (TCN)**

En esta notebook se introduce el uso de **Temporal Convolutional Networks (TCN)** como modelo base para la predicción de series temporales intradía.

Un TCN es una arquitectura neuronal basada en **convoluciones 1D causales y dilatadas**, diseñada para modelar secuencias temporales manteniendo el orden cronológico de la información. A diferencia de los modelos recurrentes, las TCN procesan la secuencia de forma completamente paralela, lo que mejora la estabilidad del entrenamiento y la eficiencia computacional.

Las principales características que motivan su uso en este proyecto son:

- Capacidad para capturar **dependencias temporales de corto y largo plazo**.
- **Causalidad estricta**, evitando cualquier fuga de información futura.
- Entrenamiento más **estable y reproducible** que modelos recurrentes clásicos.
- Buena adecuación a esquemas **seq2seq** para predicción multi-paso.

Por estas razones, el TCN se adopta como uno de los modelos principales para evaluar la capacidad predictiva sobre datos intradía del índice MNQ.



# **BLOQUE DE EJECUCIÓN COMPLETO**

## **1. Imports + paths**

In [1]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

In [2]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:

#VENTANAS 60MIN
IN_WINDOW_TRAIN_60_Z = Path(os.environ.get("IN_WINDOW_TRAIN_60_Z", "data/windows_seq2one/train_delta60_ws60.npz"))
IN_WINDOW_VALID_60_Z = Path(os.environ.get("IN_WINDOW_VALID_60_Z", "data/windows_seq2one/test_delta60_ws60.npz"))
IN_WINDOW_TEST_60_Z = Path(os.environ.get("IN_WINDOW_TEST_60_Z", "data/windows_seq2one/valid_delta60_ws60.npz"))

#VENTANAS 90MIN
IN_WINDOW_TRAIN_90_Z = Path(os.environ.get("IN_WINDOW_TRAIN_90_Z", "data/windows_seq2one/train_delta90_ws60.npz"))
IN_WINDOW_VALID_90_Z = Path(os.environ.get("IN_WINDOW_VALID_90_Z", "data/windows_seq2one/test_delta90_ws60.npz"))
IN_WINDOW_TEST_90_Z = Path(os.environ.get("IN_WINDOW_TEST_90_Z", "data/windows_seq2one/valid_delta90_ws60.npz"))

#ESCALADOR GLOBAL
IN_SCALER = Path(os.environ.get("IN_SCALER", "data/scaled/scaler.joblib"))



In [3]:
#ARTIFACTS

# Summary del stage_03a (donde está delta_target_p70 por horizonte).
#IN_TARGET_INVESTIGATION_SUMMARY = Path(os.environ.get("IN_TARGET_INVESTIGATION_SUMMARY", "reports/stage_03a_target_investigation_summary.json"))

# Summary del stage_06 (donde está window_size y n_features por horizonte).
#IN_WINDOWS_SCALING_SUMMARY =Path(os.environ.get("IN_WINDOWS_SCALING_SUMMARY", "reports/stage_06_window_scaling_seq2seq_summary.json"))

OUT_MODEL_METRICS = Path(os.environ.get("OUT_MODEL_METRICS", f"reports/stage_07__model_metrics.json"))

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


In [5]:
#PARA LA NOTEBOOK
IN_WINDOW_TRAIN_60_Z = DRIVE_DIR / IN_WINDOW_TRAIN_60_Z
IN_WINDOW_VALID_60_Z = DRIVE_DIR / IN_WINDOW_VALID_60_Z
IN_WINDOW_TEST_60_Z = DRIVE_DIR / IN_WINDOW_TEST_60_Z

IN_WINDOW_TRAIN_90_Z = DRIVE_DIR / IN_WINDOW_TRAIN_90_Z
IN_WINDOW_VALID_90_Z=DRIVE_DIR / IN_WINDOW_VALID_90_Z
IN_WINDOW_TEST_90_Z = DRIVE_DIR / IN_WINDOW_TEST_90_Z

IN_SCALER = DRIVE_DIR / IN_SCALER

#IN_WINDOWS_SCALING_SUMMARY = DRIVE_DIR / IN_WINDOWS_SCALING_SUMMARY
#IN_TARGET_INVESTIGATION_SUMMARY = DRIVE_DIR / IN_TARGET_INVESTIGATION_SUMMARY

## **2. Reproducibilidad**

In [6]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **3. Configuración**

In [7]:
def _read_json(path: Path) -> Dict[str, Any]:
    """Lee un JSON y devuelve un dict Python (con validación básica de existencia)."""
    # Verifica que el archivo exista antes de abrirlo.
    if not path.exists():
        # Si no existe, corta la ejecución con un error claro.
        raise FileNotFoundError(f"No existe el JSON: {path}")
    # Abre el archivo en modo lectura, asegurando UTF-8.
    with path.open("r", encoding="utf-8") as f:
        # Parsea el contenido JSON y lo devuelve como dict.
        return json.load(f)

In [8]:
# Config final para entrenar (modelo, horizonte).
@dataclass(frozen=True)
class StageConfig:
    # Horizonte (60 o 90).
    horizon: int
    # Largo de ventana (timesteps) desde stage_06.
    seq_len: int
    # Cantidad de features desde stage_06 para ese horizonte.
    n_features: int
    # Nombres de features (orden exacto) para ese horizonte.
    feature_names: List[str]
    # Nombre del target para ese horizonte.
    target_name: List[str]
    # Umbral mínimo económico (DELTA_BASE).
    delta_base: float
    # Umbral de oportunidad (DELTA_OP) leído del stage_03a.
    delta_op: float

In [9]:
# Construye un StageConfig leyendo ambos reports.
SUMMARY = """
def load_state_from_reports(
    horizon: int,
    #model_name: str,
    *,
    in_windows_scaling_summary: Path = IN_WINDOWS_SCALING_SUMMARY,
    in_target_investigation_summary: Path = IN_TARGET_INVESTIGATION_SUMMARY,
) -> StageConfig:
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Carga JSON del stage_06.
    w = _read_json(in_windows_scaling_summary)

    # Carga JSON del stage_03a.
    t = _read_json(in_target_investigation_summary)

    # Lee window_size global (SEQ_LEN).
    seq_len = int(w["details"]["config"]["window_size"])

    # Selecciona dataset del horizonte (ojo: "60" o "90" como string).
    ds = w["details"]["datasets"][str(horizon)]

    # Lee n_features del horizonte.
    n_features = int(ds["n_features"])

    # Lee feature_names del horizonte (aquí se refleja el 1 feature distinto).
    feature_names = list(ds["feature_names"])

    # Lee target del horizonte.
    target_name = ds["target"]

    # Valida consistencia.
    if len(feature_names) != n_features:
        raise ValueError("Inconsistencia entre n_features y feature_names")

    # Define la key de delta_base_med del stage_03a.
    target_base = f"h{horizon}_delta_base_med"
    delta_base = float(t["metrics"][target_base])

    # Define la key de delta_target_p70 del stage_03a.
    target_op = f"h{horizon}_delta_target_p70"
    # Lee DELTA_OP para ese horizonte.
    delta_op = float(t["metrics"][target_op])

    # Devuelve la config lista para entrenar.
    return StageConfig(
        horizon=horizon,
        seq_len=seq_len,
        n_features=n_features,
        feature_names=feature_names,
        target_name=target_name,
        delta_base=float(delta_base),
        delta_op=float(delta_op),
            )
"""

In [10]:
#states_h60 = load_state_from_reports(horizon=60)
#states_h60

In [11]:
#states_h90 = load_state_from_reports(horizon=90)
#states_h90


## **4. Importar métricas comunes desde .py**

In [12]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [13]:
print(compute_seq2one_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    


## **5. Carga de data windows**

In [14]:
# --------------------------------------------------
# Función común: carga .npz estándar (X, y)
# --------------------------------------------------
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.

    Espera claves:
    - 'X': array (n_samples, seq_len, n_features)
    - 'y' o 'Y': array (n_samples, seq_len) o (n_samples, seq_len, 1)
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Carga el NPZ (lectura).
    data = np.load(path)

    # Lee X (obligatoria).
    if "X" not in data:
        raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
    X = data["X"]

    # Lee y: soporta 'y' (convención usada) o 'Y' (por compatibilidad).
    if "y" in data:
        y = data["y"]
    elif "Y" in data:
        y = data["Y"]
    else:
        raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

    # Devuelve X e y.
    return X, y

In [15]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [16]:
# Cargar scaler
#scaler = joblib.load("/content/drive/MyDrive/neural_profit/data/scaled/scaler.joblib")

In [17]:
# --------------------------------------------------
# Carga completa: train/valid/test + scaler por horizonte
# --------------------------------------------------
def load_windows_and_scaler_for_horizon(horizon: int) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler para un horizonte dado (60 o 90).

    Retorna un dict:
    {
      "horizon": 60,
      "paths": {...},
      "scaler": <StandardScaler>,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }
    """
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Selecciona paths según horizonte.
    if horizon == 60:
        train_path = IN_WINDOW_TRAIN_60_Z
        valid_path = IN_WINDOW_VALID_60_Z
        test_path  = IN_WINDOW_TEST_60_Z
        scaler_path = IN_SCALER
    else:
        train_path = IN_WINDOW_TRAIN_90_Z
        valid_path = IN_WINDOW_VALID_90_Z
        test_path  = IN_WINDOW_TEST_90_Z
        scaler_path = IN_SCALER

    # Carga ventanas.
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    # Carga scaler.
    scaler = load_scaler(scaler_path)

    # Retorna todo empaquetado.
    return {
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [18]:
# --------------------------------------------------
# Carga efectiva: H60 y H90 (dos datasets distintos)
# --------------------------------------------------

# Carga todo para 60 min.
bundle_60 = load_windows_and_scaler_for_horizon(60)

# Carga todo para 90 min.
bundle_90 = load_windows_and_scaler_for_horizon(90)


In [19]:
# --------------------------------------------------
# Verificación rápida
# --------------------------------------------------

# Shapes H60.
print("H60 Train:", bundle_60["train"]["X"].shape, bundle_60["train"]["y"].shape)
print("H60 Valid:", bundle_60["valid"]["X"].shape, bundle_60["valid"]["y"].shape)
print("H60 Test :", bundle_60["test"]["X"].shape,  bundle_60["test"]["y"].shape)

# Shapes H90.
print("H90 Train:", bundle_90["train"]["X"].shape, bundle_90["train"]["y"].shape)
print("H90 Valid:", bundle_90["valid"]["X"].shape, bundle_90["valid"]["y"].shape)
print("H90 Test :", bundle_90["test"]["X"].shape,  bundle_90["test"]["y"].shape)

# Información útil (scaler).
print("Scaler H60:", type(bundle_60["scaler"]).__name__)
print("Scaler H90:", type(bundle_90["scaler"]).__name__)

H60 Train: (330144, 1200) (330144,)
H60 Valid: (70952, 1200) (70952,)
H60 Test : (70590, 1200) (70590,)
H90 Train: (330144, 1200) (330144,)
H90 Valid: (70952, 1200) (70952,)
H90 Test : (70590, 1200) (70590,)
Scaler H60: StandardScaler
Scaler H90: StandardScaler


NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **6. Sanity Check**

In [20]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [21]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [22]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [23]:
summary = run_sanity_checks_all_horizons_seq2one(bundle_60, bundle_90)

#summary["h60"]["train"]

[sanity_check_seq2one] train_h60 | X=(330144, 1200) | y=(330144,) | mode=2d | y_std=57.959399
[sanity_check_seq2one] valid_h60 | X=(70952, 1200) | y=(70952,) | mode=2d | y_std=92.974502
[sanity_check_seq2one] test_h60 | X=(70590, 1200) | y=(70590,) | mode=2d | y_std=54.347220
OK h60 (h=60)
[sanity_check_seq2one] train_h90 | X=(330144, 1200) | y=(330144,) | mode=2d | y_std=70.925909
[sanity_check_seq2one] valid_h90 | X=(70952, 1200) | y=(70952,) | mode=2d | y_std=114.837946
[sanity_check_seq2one] test_h90 | X=(70590, 1200) | y=(70590,) | mode=2d | y_std=67.180748
OK h90 (h=90)


# **DEFINICIÓN DE MODELO**

## **7. Definición del modelo — placeholder**

### **7.1. Modelo TCN**

El **Temporal Convolutional Network (TCN)** es una arquitectura neuronal diseñada para modelar dependencias temporales en secuencias mediante **convoluciones 1D causales y dilatadas**, preservando estrictamente el orden temporal de la información.

A diferencia del MLP, el TCN no aplana la ventana histórica, sino que procesa la secuencia respetando su estructura temporal. En contraste con los modelos recurrentes (LSTM / GRU), el TCN no utiliza estados ocultos ni recurrencia explícita, sino que captura dependencias de largo alcance aumentando el **campo receptivo** a través de la dilatación.

En configuración **many-to-one**, el modelo utiliza la representación temporal final (último paso o pooling temporal) como resumen de toda la ventana para predecir un valor escalar futuro.

---

**Idea básica**

El TCN se construye a partir de bloques convolucionales causales, cada uno compuesto por:

- Convolución 1D causal dilatada  
- Activación no lineal  
- (Opcional) normalización  
- Conexión residual  

La dilatación permite que el modelo incorpore información de pasos temporales lejanos sin incrementar significativamente la profundidad.

Formalmente, una capa convolucional dilatada se define como:

$$
y_t = \sum_{k=0}^{K-1} w_k \, x_{t - d \cdot k}
$$

donde:

- $ x_t \in \mathbb{R}^{20} $ es el vector de features en el minuto $ t $,
- $ K $ es el tamaño del kernel,
- $ d $ es el factor de dilatación,
- la causalidad garantiza que $x_{t'}$ con $ t' > t $ no sea utilizada.

La salida final del bloque TCN, correspondiente al último instante temporal $ T $, se proyecta mediante una capa lineal:

$$
\hat{y} = W_o h_T + b_o
$$

donde $ h_T $ resume toda la ventana histórica (por ejemplo, 60 minutos).

---

**Regularización (TCN)**

Riesgo: **Medio**, controlado principalmente por la arquitectura.

Mecanismos principales:
- **Early stopping**: principal control de sobreajuste.
- **Profundidad moderada**: número limitado de bloques.
- **Dimensión de canales controlada**.
- **Dropout (opcional)**:
  - aplicado dentro de los bloques convolucionales,
  - no afecta la causalidad.

En general, el TCN presenta un entrenamiento más estable que los modelos recurrentes y requiere menos ajustes finos de regularización.

---

**Por qué el TCN es relevante en este proyecto**

- Entrada secuencial explícita: **60 x 20**.
- Capacidad para capturar:
  - dependencias temporales de corto, mediano y largo plazo,
  - estructura intradía sin recurrencia.
- Modelo:
  - completamente paralelizable,
  - estable en entrenamiento,
  - adecuado para esquemas **seq2seq** y **many-to-one**.

El TCN actúa como una alternativa no recurrente robusta frente a LSTM y GRU, permitiendo evaluar si una arquitectura convolucional logra un mejor compromiso entre desempeño, estabilidad y eficiencia.

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- Tipo: TCN many-to-one  
- Número de bloques: bajo (por ejemplo, 3–5)  
- Tamaño del kernel: pequeño (2–5)  
- Canales por bloque: moderados  
- Dilataciones: crecientes (potencias de 2)  
- Dropout: desactivado inicialmente  
- Optimización: Adam  
- Early stopping: activado  
- Evaluación externa sobre VALID  

El ajuste fino de la arquitectura se aborda en etapas posteriores.


A continuación tiene una implementación TCN many-to-one con tuning de hiperparámetros, siguiendo una estructura típica y reutilizable (dataset por bundle, entrenamiento, validación externa, Optuna).

Supuesto razonable (alineado a notebooks anteriores): ya dispone de bundle con `X_train`, `y_train`, `X_valid`, `y_valid` (y opcional `X_test`, `y_test`), donde:

- `X` tiene shape (`N`, `seq_len`, `n_features`)
- `y` tiene shape (`N`,) o (`N`, `1`)

### **7.2. Imports (PyTorch) + semillas**

In [57]:
!pip install optuna

In [58]:
from __future__ import annotations

import math
import time
from dataclasses import dataclass
from typing import Dict, Any, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# (Opcional) Optuna
import optuna

### **7.3. Utilidades**

In [59]:
def set_seed(seed: int = 42) -> None:
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def to_1d(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y)
    if y.ndim == 2 and y.shape[1] == 1:
        return y[:, 0]
    return y


### **7.4. DataLoaders desde bundle (con reshape interno)**

In [60]:
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import torch

def _ensure_3d_X(X: np.ndarray, n_features: int) -> np.ndarray:
    X = np.asarray(X)
    if X.ndim == 3:
        return X
    if X.ndim != 2:
        raise ValueError(f"X debe ser 2D o 3D. Recibido: {X.shape}")
    n, d = X.shape
    if d % n_features != 0:
        raise ValueError(f"No puedo reshape: d={d} no es múltiplo de n_features={n_features}")
    seq_len = d // n_features
    return X.reshape(n, seq_len, n_features)

def make_loaders_from_bundle(
    bundle,
    *,
    batch_size: int,
    n_features: int,          # <- NUEVO
    num_workers: int = 0,
    pin_memory: bool = True,
):
    X_train = _ensure_3d_X(bundle["train"]["X"], n_features)
    y_train = np.asarray(bundle["train"]["y"]).reshape(-1)

    X_valid = _ensure_3d_X(bundle["valid"]["X"], n_features)
    y_valid = np.asarray(bundle["valid"]["y"]).reshape(-1)

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)
    X_valid_t = torch.tensor(X_valid, dtype=torch.float32)
    y_valid_t = torch.tensor(y_valid, dtype=torch.float32)

    dl_train = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=pin_memory)
    dl_valid = DataLoader(TensorDataset(X_valid_t, y_valid_t), batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=pin_memory)
    return dl_train, dl_valid

### **7.5. TCN core: TemporalBlock + TCNRegressor (many-to-one)**

In [61]:
class Chomp1d(nn.Module):
    """Recorta padding a la derecha para mantener causalidad."""
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = int(chomp_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T)
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        *,
        kernel_size: int,
        dilation: int,
        dropout: float,
    ):
        super().__init__()
        k = int(kernel_size)
        d = int(dilation)
        pad = (k - 1) * d  # padding causal (a la izquierda, se logra con padding simétrico + chomp)

        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=k, dilation=d, padding=pad)
        self.chomp1 = Chomp1d(pad)
        self.act1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=k, dilation=d, padding=pad)
        self.chomp2 = Chomp1d(pad)
        self.act2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)

        self.downsample = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else None
        self.out_act = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T)
        y = self.conv1(x)
        y = self.chomp1(y)
        y = self.act1(y)
        y = self.drop1(y)

        y = self.conv2(y)
        y = self.chomp2(y)
        y = self.act2(y)
        y = self.drop2(y)

        res = x if self.downsample is None else self.downsample(x)
        return self.out_act(y + res)


class TCNRegressor(nn.Module):
    """
    Many-to-one:
      input  X: (B, T, F)
      output y: (B,)
    """
    def __init__(
        self,
        n_features: int,
        *,
        channels: Tuple[int, ...],
        kernel_size: int,
        dropout: float,
        use_layernorm: bool = False,
    ):
        super().__init__()
        self.n_features = int(n_features)

        blocks = []
        in_ch = self.n_features
        for i, out_ch in enumerate(channels):
            dilation = 2 ** i
            blocks.append(
                TemporalBlock(
                    in_ch, int(out_ch),
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = int(out_ch)

        self.tcn = nn.Sequential(*blocks)
        self.use_layernorm = bool(use_layernorm)
        self.ln = nn.LayerNorm(in_ch) if self.use_layernorm else None

        self.head = nn.Linear(in_ch, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F) -> (B, F, T)
        x = x.transpose(1, 2)
        h = self.tcn(x)          # (B, C, T)
        h_last = h[:, :, -1]     # (B, C)

        if self.ln is not None:
            h_last = self.ln(h_last)

        y = self.head(h_last).squeeze(-1)  # (B,)
        return y

### **7.6. Train / eval loop (Early stopping)**



In [69]:
# ============================================================
# 2) Train / eval loop (Early stopping)
# ============================================================

@dataclass
class TrainConfig:
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    epochs: int = 30
    patience: int = 6
    grad_clip: float = 1.0
    log_every: int = 200


@torch.no_grad()
def evaluate_mae_rmse(model: nn.Module, dl: DataLoader, device: str) -> Dict[str, float]:
    model.eval()
    y_true_all, y_pred_all = [], []
    for xb, yb in dl:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        yp = model(xb)
        y_true_all.append(yb.detach().cpu().numpy())
        y_pred_all.append(yp.detach().cpu().numpy())
    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)

    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    return {"MAE": mae, "RMSE": rmse}


def train_one_trial(
    model: nn.Module,
    dl_train: DataLoader,
    dl_valid: DataLoader,
    *,
    lr: float,
    weight_decay: float,
    cfg: TrainConfig,
) -> Dict[str, Any]:
    device = cfg.device
    model.to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_mae = float("inf")
    best_state = None
    bad = 0

    step = 0
    t0 = time.time()
    for ep in range(cfg.epochs):
        model.train()
        for xb, yb in dl_train:
            step += 1
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            yp = model(xb)
            loss = F.smooth_l1_loss(yp, yb)  # Huber / SmoothL1 (robusta a outliers)
            loss.backward()

            if cfg.grad_clip is not None and cfg.grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

            opt.step()

        metrics = evaluate_mae_rmse(model, dl_valid, device)
        mae = metrics["MAE"]

        if mae < best_mae:
            best_mae = mae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break

    elapsed = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    out = {
        "best_valid_MAE": float(best_mae),
        "final_valid": evaluate_mae_rmse(model, dl_valid, device),
        "elapsed_sec": float(elapsed),
        "epochs_ran": ep + 1,
        "best_state_dict": best_state,  # opcional: guardar afuera
    }
    return out


### **7.7. Optuna objective (tuning)**


In [70]:
# ============================================================
# 3) Optuna objective (tuning)
# ============================================================

def build_tcn_from_trial(trial: optuna.Trial, n_features: int) -> TCNRegressor:
    # Profundidad y canales
    n_blocks = trial.suggest_int("n_blocks", 3, 4)              # antes 3..6
    base_ch  = trial.suggest_categorical("base_channels", [16, 32, 48])  # antes incluía 64
    growth   = trial.suggest_categorical("channel_growth", [1.0, 1.25])  # antes 1.5


    channels = []
    ch = float(base_ch)
    for _ in range(n_blocks):
        channels.append(int(round(ch)))
        ch *= float(growth)

    kernel_size = trial.suggest_int("kernel_size", 2, 3)        # antes 2..5
    dropout = trial.suggest_float("dropout", 0.0, 0.2)          # antes 0.3
    use_ln = trial.suggest_categorical("use_layernorm", [False, True])

    return TCNRegressor(
        n_features=n_features,
        channels=tuple(channels),
        kernel_size=kernel_size,
        dropout=dropout,
        use_layernorm=use_ln,
    )


def objective(trial, *, bundle, cfg):
    X_train = np.asarray(bundle["train"]["X"])

    if X_train.ndim == 3:
        n_features = int(X_train.shape[-1])
    else:
        seq_len = int(bundle.get("seq_len", 60))  # <- H60: 60 (ajuste si su ventana es otra)
        d = int(X_train.shape[1])
        assert d % seq_len == 0, f"X 2D con d={d} no es divisible por seq_len={seq_len}"
        n_features = d // seq_len

    batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024, 2048])
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-8, 1e-3, log=True)

    dl_train, dl_valid = make_loaders_from_bundle(
        bundle,
        batch_size=batch_size,
        n_features=n_features,   # <- NUEVO
        num_workers=0,
        pin_memory=True,
    )

    model = build_tcn_from_trial(trial, n_features=n_features)

    result = train_one_trial(model, dl_train, dl_valid, lr=lr, weight_decay=weight_decay, cfg=cfg)
    return result["best_valid_MAE"]


### **7.8. Run tuning**


In [71]:
set_seed(42)

CFG = TrainConfig(
    device="cuda" if torch.cuda.is_available() else "cpu",
    epochs=12,        # antes 30
    patience=3,       # antes 6
    grad_clip=1.0,
)

# bundle_60 y/o bundle_90 deberían existir en su notebook
# Ejemplo: best_params_60, best_value_60 = run_optuna(bundle_60)

def run_optuna(bundle: Dict[str, Any], n_trials: int = 25, study_name: str = "tcn") -> Tuple[Dict[str, Any], float, optuna.Study]:
    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(direction="minimize", sampler=sampler, study_name=study_name)
    study.optimize(lambda t: objective(t, bundle=bundle, cfg=CFG), n_trials=n_trials, show_progress_bar=True)
    return study.best_params, float(study.best_value), study




### **7.9. Ejecución**


In [72]:
import torch

# Check CUDA
assert torch.cuda.is_available(), "CUDA no disponible"

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

# Check modelo en GPU
model = TCNRegressor(
    n_features=bundle_60["train"]["X"].shape[-1],
    channels=(32, 32, 32),
    kernel_size=3,
    dropout=0.0,
)
model.to(CFG.device)

assert next(model.parameters()).is_cuda, "Modelo NO está en GPU"
print("Modelo en GPU: OK")


GPU: NVIDIA L4
CUDA: 12.6
Modelo en GPU: OK


In [73]:
def subsample_train(bundle, frac=0.25, seed=42):
    rng = np.random.default_rng(seed)
    X = bundle["train"]["X"]; y = bundle["train"]["y"]
    n = len(X)
    idx = rng.choice(n, size=int(n*frac), replace=False)
    b = {**bundle}
    b["train"] = {**bundle["train"], "X": X[idx], "y": y[idx]}
    return b

bundle_60_tune = subsample_train(bundle_60, frac=0.25)
bundle_90_tune = subsample_train(bundle_90, frac=0.25)

In [74]:
best_params_60, best_mae_60, study_60 = run_optuna(bundle_60_tune, n_trials=30, study_name="tcn_h60")

[I 2026-02-01 20:48:33,410] A new study created in memory with name: tcn_h60


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-02-01 20:48:49,955] Trial 0 finished with value: 61.397037506103516 and parameters: {'batch_size': 512, 'lr': 0.0001700037298921101, 'weight_decay': 6.02521573620385e-08, 'n_blocks': 3, 'base_channels': 16, 'channel_growth': 1.25, 'kernel_size': 3, 'dropout': 0.04246782213565523, 'use_layernorm': True}. Best is trial 0 with value: 61.397037506103516.
[I 2026-02-01 20:49:01,498] Trial 1 finished with value: 61.368656158447266 and parameters: {'batch_size': 512, 'lr': 0.0008012737503998541, 'weight_decay': 4.982752357076433e-08, 'n_blocks': 3, 'base_channels': 48, 'channel_growth': 1.25, 'kernel_size': 3, 'dropout': 0.009290082543999545, 'use_layernorm': False}. Best is trial 1 with value: 61.368656158447266.
[I 2026-02-01 20:49:14,661] Trial 2 finished with value: 61.388389587402344 and parameters: {'batch_size': 1024, 'lr': 0.00028180680291847244, 'weight_decay': 3.078651783619614e-08, 'n_blocks': 4, 'base_channels': 48, 'channel_growth': 1.25, 'kernel_size': 2, 'dropout': 0.13

In [75]:
best_params_90, best_mae_90, study_90 = run_optuna(bundle_90_tune, n_trials=30, study_name="tcn_h90")

[I 2026-02-01 20:56:47,389] A new study created in memory with name: tcn_h90


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-02-01 20:57:09,062] Trial 0 finished with value: 75.44256591796875 and parameters: {'batch_size': 512, 'lr': 0.0001700037298921101, 'weight_decay': 6.02521573620385e-08, 'n_blocks': 3, 'base_channels': 16, 'channel_growth': 1.25, 'kernel_size': 3, 'dropout': 0.04246782213565523, 'use_layernorm': True}. Best is trial 0 with value: 75.44256591796875.
[I 2026-02-01 20:57:20,720] Trial 1 finished with value: 75.49141693115234 and parameters: {'batch_size': 512, 'lr': 0.0008012737503998541, 'weight_decay': 4.982752357076433e-08, 'n_blocks': 3, 'base_channels': 48, 'channel_growth': 1.25, 'kernel_size': 3, 'dropout': 0.009290082543999545, 'use_layernorm': False}. Best is trial 0 with value: 75.44256591796875.
[I 2026-02-01 20:57:31,702] Trial 2 finished with value: 75.44253540039062 and parameters: {'batch_size': 1024, 'lr': 0.00028180680291847244, 'weight_decay': 3.078651783619614e-08, 'n_blocks': 4, 'base_channels': 48, 'channel_growth': 1.25, 'kernel_size': 2, 'dropout': 0.1325044

### **7.10 Entrenamiento TCN con best params**


In [81]:
best_params_60 #, best_mae_60, study_60

{'batch_size': 256,
 'lr': 0.00011128194768838959,
 'weight_decay': 1.5207297997231788e-05,
 'n_blocks': 3,
 'base_channels': 32,
 'channel_growth': 1.25,
 'kernel_size': 2,
 'dropout': 0.0153959819657586,
 'use_layernorm': False}

{'batch_size': 256,
 'lr': 0.00011128194768838959,
 'weight_decay': 1.5207297997231788e-05,
 'n_blocks': 3,
 'base_channels': 32,
 'channel_growth': 1.25,
 'kernel_size': 2,
 'dropout': 0.0153959819657586,
 'use_layernorm': False}

In [82]:
best_params_90 #, best_mae_90, study_90

{'batch_size': 256,
 'lr': 0.00011128194768838959,
 'weight_decay': 1.5207297997231788e-05,
 'n_blocks': 3,
 'base_channels': 32,
 'channel_growth': 1.25,
 'kernel_size': 2,
 'dropout': 0.0153959819657586,
 'use_layernorm': False}

{'batch_size': 256,
 'lr': 0.00011128194768838959,
 'weight_decay': 1.5207297997231788e-05,
 'n_blocks': 3,
 'base_channels': 32,
 'channel_growth': 1.25,
 'kernel_size': 2,
 'dropout': 0.0153959819657586,
 'use_layernorm': False}

In [83]:
import numpy as np
#Paso 0: confirmar shapes e inferir n_features
# Ajuste si su ventana no es 60
SEQ_LEN_60 = 60
SEQ_LEN_90 = 60  # normalmente la ventana es la misma; cambie si no aplica

def infer_n_features_from_bundle(bundle: dict, *, seq_len: int) -> int:
    X = np.asarray(bundle["train"]["X"])
    if X.ndim == 3:
        return int(X.shape[-1])
    if X.ndim == 2:
        d = int(X.shape[1])
        assert d % seq_len == 0, f"d={d} no es divisible por seq_len={seq_len}"
        return int(d // seq_len)
    raise ValueError(f"X train debe ser 2D o 3D. Recibido: {X.shape}")

# --- H60 ---
n_features_60 = infer_n_features_from_bundle(bundle_60, seq_len=SEQ_LEN_60)
print("H60 -> n_features:", n_features_60)
print("H60 train X:", np.asarray(bundle_60["train"]["X"]).shape, "y:", np.asarray(bundle_60["train"]["y"]).shape)
print("H60 valid X:", np.asarray(bundle_60["valid"]["X"]).shape, "y:", np.asarray(bundle_60["valid"]["y"]).shape)
print("H60 test  X:", np.asarray(bundle_60["test"]["X"]).shape,  "y:", np.asarray(bundle_60["test"]["y"]).shape)

# --- H90 ---
n_features_90 = infer_n_features_from_bundle(bundle_90, seq_len=SEQ_LEN_90)
print("\nH90 -> n_features:", n_features_90)
print("H90 train X:", np.asarray(bundle_90["train"]["X"]).shape, "y:", np.asarray(bundle_90["train"]["y"]).shape)
print("H90 valid X:", np.asarray(bundle_90["valid"]["X"]).shape, "y:", np.asarray(bundle_90["valid"]["y"]).shape)
print("H90 test  X:", np.asarray(bundle_90["test"]["X"]).shape,  "y:", np.asarray(bundle_90["test"]["y"]).shape)


H60 -> n_features: 20
H60 train X: (330144, 1200) y: (330144,)
H60 valid X: (70952, 1200) y: (70952,)
H60 test  X: (70590, 1200) y: (70590,)

H90 -> n_features: 20
H90 train X: (330144, 1200) y: (330144,)
H90 valid X: (70952, 1200) y: (70952,)
H90 test  X: (70590, 1200) y: (70590,)


In [84]:
#Paso 1 - Crear el bundle_final (TRAIN = TRAIN+VALID) sin tocar el original
def make_bundle_train_plus_valid(bundle: dict) -> dict:
    """
    Crea un NUEVO bundle donde:
      - train = concatenación de train + valid
      - valid se mantiene igual (por consistencia)
      - test  se mantiene igual
    Importante:
      - No modifica el bundle original (solo crea una copia).
    """
    # Copia superficial del dict principal
    out = dict(bundle)

    # Extraer arrays de train y valid
    X_tr = np.asarray(bundle["train"]["X"])
    y_tr = np.asarray(bundle["train"]["y"]).reshape(-1)

    X_va = np.asarray(bundle["valid"]["X"])
    y_va = np.asarray(bundle["valid"]["y"]).reshape(-1)

    # Concatenar (apilado por filas) para formar el nuevo TRAIN
    X_tv = np.concatenate([X_tr, X_va], axis=0)
    y_tv = np.concatenate([y_tr, y_va], axis=0)

    # Reemplazar SOLO el split train
    out["train"] = {"X": X_tv, "y": y_tv}

    # Mantener valid y test tal cual
    out["valid"] = bundle["valid"]
    out["test"]  = bundle["test"]

    return out


# --- Construir bundles finales para re-entrenamiento ---
bundle_60_final = make_bundle_train_plus_valid(bundle_60)
bundle_90_final = make_bundle_train_plus_valid(bundle_90)

# --- Checks rápidos de sanity ---
print("H60 final train:", bundle_60_final["train"]["X"].shape, bundle_60_final["train"]["y"].shape)
print("H90 final train:", bundle_90_final["train"]["X"].shape, bundle_90_final["train"]["y"].shape)


H60 final train: (401096, 1200) (401096,)
H90 final train: (401096, 1200) (401096,)


In [85]:
# ============================================================
# PASO 2) DataLoaders para re-entrenamiento final
# - Usa bundle_60_final / bundle_90_final (train = train+valid)
# - Usa n_features=20 (ya inferido)
# - Usa batch_size desde best_params
# ============================================================

# --- Constantes ya conocidas ---
N_FEATURES = 20  # (confirmado: 1200 = 60 * 20)

# ----------------------------
# H60: loaders con best_params_60
# ----------------------------
# Batch size que Optuna eligió como óptimo para H60
bs60 = int(best_params_60["batch_size"])

# Crea dl_train (train+valid) y dl_valid (valid original) con su helper existente
dl_train_60, dl_valid_60 = make_loaders_from_bundle(
    bundle_60_final,
    batch_size=bs60,
    n_features=N_FEATURES,
    num_workers=0,
    pin_memory=True,
)

# Sanity check: un batch y sus shapes
xb, yb = next(iter(dl_train_60))
print("H60 dl_train batch X:", xb.shape, "y:", yb.shape)
xbv, ybv = next(iter(dl_valid_60))
print("H60 dl_valid batch X:", xbv.shape, "y:", ybv.shape)

# ----------------------------
# H90: loaders con best_params_90
# ----------------------------
bs90 = int(best_params_90["batch_size"])

dl_train_90, dl_valid_90 = make_loaders_from_bundle(
    bundle_90_final,
    batch_size=bs90,
    n_features=N_FEATURES,
    num_workers=0,
    pin_memory=True,
)

xb, yb = next(iter(dl_train_90))
print("H90 dl_train batch X:", xb.shape, "y:", yb.shape)
xbv, ybv = next(iter(dl_valid_90))
print("H90 dl_valid batch X:", xbv.shape, "y:", ybv.shape)


H60 dl_train batch X: torch.Size([256, 60, 20]) y: torch.Size([256])
H60 dl_valid batch X: torch.Size([256, 60, 20]) y: torch.Size([256])
H90 dl_train batch X: torch.Size([256, 60, 20]) y: torch.Size([256])
H90 dl_valid batch X: torch.Size([256, 60, 20]) y: torch.Size([256])


In [86]:
# ============================================================
# PASO 3) Construir y entrenar el modelo final (H60 / H90)
# - Modelo TCN many-to-one
# - Parámetros fijos: best_params_60 / best_params_90
# - Entrena en GPU si está disponible
# ============================================================

import torch

# ----------------------------
# 0) Confirmar device (GPU)
# ----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ----------------------------
# 1) Función: construir TCN desde best_params (misma lógica del tuning)
# ----------------------------
def build_tcn_from_best_params(best_params: dict, *, n_features: int) -> TCNRegressor:
    """
    Reconstruye exactamente la arquitectura del trial ganador.
    """
    # Cantidad de bloques TCN
    n_blocks = int(best_params["n_blocks"])

    # Definir canales por bloque (base * growth^i)
    base_ch = int(best_params["base_channels"])
    growth = float(best_params["channel_growth"])

    channels = []
    ch = float(base_ch)
    for _ in range(n_blocks):
        channels.append(int(round(ch)))
        ch *= growth

    # Crear modelo
    model = TCNRegressor(
        n_features=int(n_features),
        channels=tuple(channels),
        kernel_size=int(best_params["kernel_size"]),
        dropout=float(best_params["dropout"]),
        use_layernorm=bool(best_params["use_layernorm"]),
    )
    return model

# ----------------------------
# 2) Config de entrenamiento final (puede ajustar epochs/patience)
# ----------------------------
CFG_FINAL = TrainConfig(
    device=DEVICE,   # <- GPU si existe
    epochs=30,
    patience=6,
    grad_clip=1.0,
)

# ----------------------------
# 3) Entrenamiento final H60
# ----------------------------
# 3.1) Construir modelo con best params
model_tcn_final_60 = build_tcn_from_best_params(best_params_60, n_features=20)

# 3.2) Entrenar (train+valid) con early stopping usando valid original
res_final_60 = train_one_trial(
    model_tcn_final_60,
    dl_train_60,
    dl_valid_60,
    lr=float(best_params_60["lr"]),
    weight_decay=float(best_params_60["weight_decay"]),
    cfg=CFG_FINAL,
)

print("H60 listo. Mejor valid MAE:", res_final_60["best_valid_MAE"])

# ----------------------------
# 4) Entrenamiento final H90
# ----------------------------
model_tcn_final_90 = build_tcn_from_best_params(best_params_90, n_features=20)

res_final_90 = train_one_trial(
    model_tcn_final_90,
    dl_train_90,
    dl_valid_90,
    lr=float(best_params_90["lr"]),
    weight_decay=float(best_params_90["weight_decay"]),
    cfg=CFG_FINAL,
)

print("H90 listo. Mejor valid MAE:", res_final_90["best_valid_MAE"])

# ----------------------------
# 5) Check explícito de GPU (modelo realmente en CUDA)
# ----------------------------
print("H60 model on:", next(model_tcn_final_60.parameters()).device)
print("H90 model on:", next(model_tcn_final_90.parameters()).device)


Device: cuda
H60 listo. Mejor valid MAE: 59.894744873046875
H90 listo. Mejor valid MAE: 73.57549285888672
H60 model on: cuda:0
H90 model on: cuda:0


### **7.11. Predicciones**


In [89]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

# ----------------------------
# 1) DataLoader para un split (valid o test)
# ----------------------------
def make_loader_from_split(bundle: dict, *, split: str, batch_size: int, n_features: int) -> DataLoader:
    # X puede venir 2D (N,1200): lo pasamos a 3D (N,60,20) con su helper existente
    X = _ensure_3d_X(bundle[split]["X"], n_features)
    y = np.asarray(bundle[split]["y"]).reshape(-1)

    # Convertir a tensores
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32)

    # Loader (sin shuffle para OOS)
    return DataLoader(
        TensorDataset(X_t, y_t),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

# ----------------------------
# 2) Inferencia (retorna y_true y y_pred en numpy)
# ----------------------------
@torch.no_grad()
def predict_oos(model: torch.nn.Module, dl: DataLoader, *, device: str) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    y_true_list, y_pred_list = [], []

    for xb, yb in dl:
        xb = xb.to(device, non_blocking=True)   # GPU
        yp = model(xb)                          # (B,)

        y_true_list.append(yb.numpy())
        y_pred_list.append(yp.detach().cpu().numpy())

    y_true = np.concatenate(y_true_list).reshape(-1)
    y_pred = np.concatenate(y_pred_list).reshape(-1)
    return y_true, y_pred


In [90]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_FEATURES = 20
BS60 = int(best_params_60["batch_size"])

dl_valid_60 = make_loader_from_split(bundle_60, split="valid", batch_size=BS60, n_features=N_FEATURES)
dl_test_60  = make_loader_from_split(bundle_60, split="test",  batch_size=BS60, n_features=N_FEATURES)

y_true_valid_60, y_pred_valid_60 = predict_oos(model_tcn_final_60, dl_valid_60, device=DEVICE)
y_true_test_60,  y_pred_test_60  = predict_oos(model_tcn_final_60, dl_test_60,  device=DEVICE)

print("H60 VALID:", y_true_valid_60.shape, y_pred_valid_60.shape)
print("H60 TEST :", y_true_test_60.shape,  y_pred_test_60.shape)

H60 VALID: (70952,) (70952,)
H60 TEST : (70590,) (70590,)


In [91]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_FEATURES = 20
BS90 = int(best_params_90["batch_size"])

# Loaders OOS
dl_valid_90 = make_loader_from_split(
    bundle_90,
    split="valid",
    batch_size=BS90,
    n_features=N_FEATURES,
)
dl_test_90 = make_loader_from_split(
    bundle_90,
    split="test",
    batch_size=BS90,
    n_features=N_FEATURES,
)

# Predicciones
y_true_valid_90, y_pred_valid_90 = predict_oos(
    model_tcn_final_90,
    dl_valid_90,
    device=DEVICE,
)
y_true_test_90, y_pred_test_90 = predict_oos(
    model_tcn_final_90,
    dl_test_90,
    device=DEVICE,
)

# Sanity check
print("H90 VALID:", y_true_valid_90.shape, y_pred_valid_90.shape)
print("H90 TEST :", y_true_test_90.shape,  y_pred_test_90.shape)

H90 VALID: (70952,) (70952,)
H90 TEST : (70590,) (70590,)


## **8. Métricas ML**

In [92]:
import pandas as pd

def metrics_to_df(metrics: dict, *, model: str, split: str, horizon: int) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """
    return pd.DataFrame([{
        "model": model,
        "split": split,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])


In [93]:
import pandas as pd

# ============================================================
# H=60
# ============================================================
y_valid_60 = bundle_60["valid"]["y"]
y_test_60  = bundle_60["test"]["y"]

metrics_valid_60 = compute_seq2one_metrics(y_valid_60, y_pred_valid_60, compute_r2=True)
metrics_test_60  = compute_seq2one_metrics(y_test_60,  y_pred_test_60,  compute_r2=True)

df_valid_60 = metrics_to_df(metrics_valid_60, model="tcn", split="valid", horizon=60)
df_test_60  = metrics_to_df(metrics_test_60,  model="tcn", split="test",  horizon=60)

# ============================================================
# H=90
# ============================================================
y_valid_90 = bundle_90["valid"]["y"]
y_test_90  = bundle_90["test"]["y"]

metrics_valid_90 = compute_seq2one_metrics(y_valid_90, y_pred_valid_90, compute_r2=True)
metrics_test_90  = compute_seq2one_metrics(y_test_90,  y_pred_test_90,  compute_r2=True)

df_valid_90 = metrics_to_df(metrics_valid_90, model="tcn", split="valid", horizon=90)
df_test_90  = metrics_to_df(metrics_test_90,  model="tcn", split="test",  horizon=90)

# ============================================================
# Tabla final (VALID+TEST, H60+H90)
# ============================================================
df_tcn_metrics = pd.concat([df_valid_60, df_valid_90, df_test_60, df_test_90], ignore_index=True)
df_tcn_metrics = df_tcn_metrics.sort_values(["split", "horizon_min"]).reset_index(drop=True)

df_tcn_metrics


,model,split,horizon_min,MAE,RMSE,R2,DA
0,tcn,test,60,40.189733,54.989275,-0.023767,0.522467
1,tcn,test,90,49.231791,68.219137,-0.031152,0.513502
2,tcn,valid,60,59.894752,91.252188,0.036706,0.563528
3,tcn,valid,90,73.575496,113.052709,0.030850,0.562107


## **9. Guardar artefactos para Stage_08**

In [94]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"metrics_{name}.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path


In [95]:
save_seq2one_metrics(
    df_tcn_metrics,
    name="tcn_tuned",
)

[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/metrics_tcn_tuned.parquet


PosixPath('/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/metrics_tcn_tuned.parquet')

## **10. Resultados y conclusiones parciales**

| # | model | split | horizon_min | MAE       | RMSE       | R2        | DA       |
|---|-------|-------|-------------|-----------|------------|-----------|----------|
| 0 | lstm  | test  | 60          | 40.062974 | 54.359750  | -0.000461 | 0.475432 |
| 1 | lstm  | valid | 60          | 61.611558 | 92.909564  | 0.001396  | 0.484279 |
| 2 | lstm  | test  | 90          | 48.837803 | 67.269610  | -0.002647 | 0.463057 |
| 3 | lstm  | valid | 90          | 75.718288 | 114.708905 | 0.002246  | 0.484445 |

| # | model | split | horizon_min | MAE       | RMSE       | R2        | DA       |
|---|-------|-------|-------------|-----------|------------|-----------|----------|
| 0 | tcn   | test  | 60          | 40.189733 | 54.989275  | -0.023767 | 0.522467 |
| 1 | tcn   | test  | 90          | 49.231791 | 68.219137  | -0.031152 | 0.513502 |
| 2 | tcn   | valid | 60          | 59.894752 | 91.252188  | 0.036706  | 0.563528 |
| 3 | tcn   | valid | 90          | 73.575496 | 113.052709 | 0.030850  | 0.562107 |


**Comparación LSTM vs TCN**

1) Error (MAE / RMSE)
    - LSTM y TCN presentan **errores muy similares en TEST** para ambos horizontes (60 y 90 minutos).
    - El TCN **no mejora el error absoluto** frente al LSTM; incluso es **ligeramente peor** en H60 y H90.

2) R² (capacidad explicativa)
    - En TEST, **ambos modelos muestran R² cercano a 0 o negativo**, lo que indica **ausencia de poder explicativo fuera de muestra**.
    - En VALID, el TCN alcanza un **R² positivo (~0.03)**, pero **no logra generalizar** este comportamiento a TEST.

3) Direccionalidad (DA)
    - El TCN **mejora de forma clara la Direccional Accuracy (DA)** respecto al LSTM:
      - H60 TEST: ~0.52 vs ~0.48  
      - H90 TEST: ~0.51 vs ~0.46
    - Esto sugiere que el TCN **capta mejor el signo del movimiento**, aunque **no la magnitud**.

4) Conclusión operativa
    - **LSTM**: desempeño similar en error, **peor direccionalidad**.
    - **TCN**: misma precisión en puntos, pero **mejor señal direccional**.


**Lectura final:** el TCN **no aporta valor adicional como modelo de regresión pura**, pero **sí resulta más prometedor como modelo direccional**, potencialmente útil como **filtro de dirección** más que como predictor de magnitud.
